# Probability

Probability is the language of uncertainty. In finance, it helps us reason about return distributions, downside risk, and the likelihood of different outcomes.

Abbreviations used in this notebook:

- **PDF**: Probability Density Function.
- **CDF**: Cumulative Distribution Function.
- **VaR**: Value at Risk.
- **CVaR**: Conditional Value at Risk.
- **CHF**: Swiss franc, used only as an illustrative currency.

## 1. Intuition

Investing decisions are made before the future is known. Probability helps us move from single-point forecasts to ranges of outcomes.

A return distribution can tell us the average outcome, the spread around that average, and the probability of losses beyond a threshold.

## 2. Mathematics

Expected value:

$$
E[X] = \sum_i p_i x_i
$$

Where:

- $E[X]$ = expected value of random variable $X$
- $p_i$ = probability of outcome $i$
- $x_i$ = value of outcome $i$

Variance:

$$
Var(X) = E[(X - E[X])^2]
$$

Where:

- $E[X]$ = expected value of random variable $X$
- $Var(X)$ = variance of random variable $X$

Standard deviation:

$$
\sigma = \sqrt{Var(X)}
$$

Where:

- $Var(X)$ = variance of random variable $X$
- $\sigma$ = standard deviation or volatility

Historical VaR at 5 percent:

$$
VaR_{5\%} = -Percentile(R, 5\%)
$$

Where:

- $VaR$ = value at risk, a downside loss threshold
- $Percentile(R, 5\%)$ = fifth percentile of the return distribution

Conditional VaR:

$$
CVaR_{5\%} = -E[R | R \le Percentile(R, 5\%)]
$$

Where:

- $VaR$ = value at risk, a downside loss threshold
- $CVaR$ = conditional value at risk
- $Percentile(R, 5\%)$ = fifth percentile of the return distribution

## 3. Implementation

We use synthetic daily returns to estimate probabilities and downside risk from observed outcomes.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "04_quantitative_methods" / "quant_utils.py"
spec = importlib.util.spec_from_file_location("quant_utils", helper_path)
quant_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(quant_utils)

plt.style.use("seaborn-v0_8-whitegrid")
returns = quant_utils.generate_return_sample()

asset = returns["asset"]
summary = pd.Series({
    "mean_daily_return": asset.mean(),
    "daily_volatility": asset.std(),
    "probability_of_loss": (asset < 0).mean(),
    "probability_loss_worse_than_2pct": (asset < -0.02).mean(),
    "daily_var_5pct": -asset.quantile(0.05),
    "daily_cvar_5pct": -asset[asset <= asset.quantile(0.05)].mean(),
})
summary.to_frame("value")

In [ ]:
outcomes = pd.DataFrame({
    "scenario": ["Bear", "Base", "Bull"],
    "probability": [0.25, 0.50, 0.25],
    "one_year_return": [-0.18, 0.07, 0.24],
})
outcomes["weighted_return"] = outcomes["probability"] * outcomes["one_year_return"]
expected_return = outcomes["weighted_return"].sum()
outcomes

## 4. Visualization

Histograms and cumulative distributions make probability visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
asset.hist(bins=40, ax=axes[0], color="#2f6f8f", edgecolor="white")
axes[0].axvline(asset.quantile(0.05), color="#9a6b2f", linestyle="--", label="5% tail")
axes[0].set_title("Daily Return Distribution")
axes[0].set_xlabel("Daily return")
axes[0].xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
axes[0].legend()

sorted_returns = np.sort(asset)
cdf = np.arange(1, len(sorted_returns)+1) / len(sorted_returns)
axes[1].plot(sorted_returns, cdf, color="#2f6f8f")
axes[1].set_title("Empirical CDF")
axes[1].set_xlabel("Daily return")
axes[1].set_ylabel("Cumulative probability")
axes[1].xaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")
plt.tight_layout(); plt.show()

## 5. Application

Probability supports risk management by estimating how often losses happen and how severe tail losses can be. It is also the basis for Monte Carlo simulation and scenario analysis.

In [ ]:
print(f"Scenario expected return: {expected_return:.1%}")
print(f"Historical probability of a negative daily return: {(asset < 0).mean():.1%}")
print(f"Historical daily CVaR at 5%: {-asset[asset <= asset.quantile(0.05)].mean():.2%}")

## 6. Reflection

- Probability turns uncertainty into structured reasoning.
- Tail losses matter more than average days for risk management.
- Historical probabilities may not match future probabilities.
- Expected value can hide wide differences in possible outcomes.

Questions to answer after running the notebook:

1. How often did the asset lose money?
2. What does CVaR add beyond VaR?
3. Why can expected return be misleading?
4. What could make historical probabilities unreliable?